# 01 — Database Setup & Schema Exploration

## What This Notebook Does
Connect to the SQLite database using PySpark,
explore the schema, understand table relationships,
and write exploratory SQL queries.

## Why PySpark Instead of Pandas
Our database has 33+ million transaction rows.
Pandas loads everything into RAM — on a laptop with
8-16GB RAM, loading 33 million rows would be slow
or crash the kernel.

PySpark uses LAZY evaluation — it builds a query plan
but does NOT load data into memory until you explicitly
ask for it. This is the fundamental difference.



In [1]:
pip install pyspark

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
from pyspark.sql import SparkSession
import os

JAR_PATH = os.path.abspath("../jars/sqlite-jdbc-3.45.1.0.jar")

DB_PATH = r"D:\some\other\drive\finance_clv.db"
DB_URL = f"jdbc:sqlite:{DB_PATH}"

spark = (
    SparkSession.builder
    .appName("CLV_Database_Setup")
    .master("local[2]")
    .config("spark.driver.memory", "4g")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.driver.extraClassPath", JAR_PATH)
    .config("spark.executor.extraClassPath", JAR_PATH)
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")

print(f"Spark version : {spark.version}")
print(f"JAR path      : {JAR_PATH}")
print(f"DB URL        : {DB_URL}")
print("SQLite JDBC driver test:")

spark.sparkContext._jvm.java.lang.Class.forName("org.sqlite.JDBC")

print("SQLite JDBC driver loaded successfully!")

c:\Users\dsp96\Desktop\realworld-ds-ml\venv\Lib\site-packages\pyspark\testing\utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


Spark version : 4.2.0
JAR path      : c:\Users\dsp96\Desktop\realworld-ds-ml\finance\personal_finance_clv\jars\sqlite-jdbc-3.45.1.0.jar
DB URL        : jdbc:sqlite:D:\some\other\drive\finance_clv.db
SQLite JDBC driver test:
SQLite JDBC driver loaded successfully!


In [3]:
def read_table(table_name):
    """
    Read a table from SQLite into a Spark DataFrame.
    This is the standard pattern we will use throughout
    all Finance notebooks.
    """
    return (
        spark.read
        .format("jdbc")
        .option("url", DB_URL)
        .option("dbtable", table_name)
        .option("driver", "org.sqlite.JDBC")
        .load()
    )

# Read the customers table
customers = read_table("customers")

print(f"Type: {type(customers)}")
print(f"This is NOT a pandas DataFrame.")
print(f"No data has been loaded into memory yet.")
print(f"Spark has only read the schema from the database.")

Type: <class 'pyspark.sql.classic.dataframe.DataFrame'>
This is NOT a pandas DataFrame.
No data has been loaded into memory yet.
Spark has only read the schema from the database.


In [4]:
def read_table(table_name):
    """
    Standard pattern to read any table from SQLite
    into a Spark DataFrame. Reused across all notebooks.
    """
    return (
        spark.read
        .format("jdbc")
        .option("url", DB_URL)
        .option("dbtable", table_name)
        .option("driver", "org.sqlite.JDBC")
        .load()
    )

# Read all five tables
customers    = read_table("customers")
products     = read_table("products")
txn_history  = read_table("transactions_history")
txn_label    = read_table("transactions_label")
clv_labels   = read_table("clv_labels")

print("All tables loaded as Spark DataFrames ✅")
print(f"\nObject types:")
print(f"  customers   : {type(customers)}")
print(f"  txn_history : {type(txn_history)}")
print(f"\nNOTE: No data is in memory yet.")
print(f"Spark only read the schema from SQLite.")
print(f"Data loads only when you call .show() or .count()")

All tables loaded as Spark DataFrames ✅

Object types:
  customers   : <class 'pyspark.sql.classic.dataframe.DataFrame'>
  txn_history : <class 'pyspark.sql.classic.dataframe.DataFrame'>

NOTE: No data is in memory yet.
Spark only read the schema from SQLite.
Data loads only when you call .show() or .count()


In [5]:
print("=" * 55)
print(" SCHEMA INSPECTION — All 5 Tables")
print("=" * 55)

tables = {
    "customers"           : customers,
    "products"            : products,
    "transactions_history": txn_history,
    "transactions_label"  : txn_label,
    "clv_labels"          : clv_labels,
}

for name, df in tables.items():
    print(f"\n── {name.upper()} ──────────────────────────")
    df.printSchema()

 SCHEMA INSPECTION — All 5 Tables

── CUSTOMERS ──────────────────────────
root
 |-- customer_id: string (nullable = true)
 |-- age: integer (nullable = true)
 |-- gender: string (nullable = true)
 |-- city: string (nullable = true)
 |-- city_tier: string (nullable = true)
 |-- segment: string (nullable = true)
 |-- monthly_income: double (nullable = true)
 |-- account_open_date: string (nullable = true)
 |-- account_age_days: integer (nullable = true)
 |-- kyc_complete: integer (nullable = true)
 |-- pan_linked: integer (nullable = true)
 |-- aadhaar_linked: integer (nullable = true)
 |-- cibil_score: double (nullable = true)
 |-- digital_score: double (nullable = true)
 |-- rm_assigned: integer (nullable = true)
 |-- is_nri: integer (nullable = true)
 |-- occupation: string (nullable = true)
 |-- phone: string (nullable = true)
 |-- email: string (nullable = true)


── PRODUCTS ──────────────────────────
root
 |-- product_id: string (nullable = true)
 |-- customer_id: string (nullabl

In [6]:
print("=" * 55)
print(" ROW COUNTS — First Real Spark Action")
print("=" * 55)
print("(This triggers actual data reading for the first time)\n")

for name, df in tables.items():
    count = df.count()
    print(f"  {name:<30}: {count:>12,} rows")

print(f"\nTotal rows across all tables:")
total = sum(df.count() for df in tables.values())
print(f"  {total:>12,} rows")

 ROW COUNTS — First Real Spark Action
(This triggers actual data reading for the first time)

  customers                     :       50,250 rows
  products                      :      124,865 rows
  transactions_history          :   16,608,362 rows
  transactions_label            :   16,608,069 rows
  clv_labels                    :       50,000 rows

Total rows across all tables:
    33,441,546 rows


In [7]:
print("CUSTOMERS TABLE — First 5 rows")
print("=" * 55)
customers.show(5, truncate=False)

print("\nCLV LABELS TABLE — First 5 rows")
print("=" * 55)
clv_labels.show(5, truncate=False)

print("\nTRANSACTIONS HISTORY — First 5 rows")
print("=" * 55)
txn_history.show(5, truncate=True)

CUSTOMERS TABLE — First 5 rows
+-----------+---+------+-------+---------+---------------+--------------+-----------------+----------------+------------+----------+--------------+-----------+-------------+-----------+------+---------------+-------------+----------------------+
|customer_id|age|gender|city   |city_tier|segment        |monthly_income|account_open_date|account_age_days|kyc_complete|pan_linked|aadhaar_linked|cibil_score|digital_score|rm_assigned|is_nri|occupation     |phone        |email                 |
+-----------+---+------+-------+---------+---------------+--------------+-----------------+----------------+------------+----------+--------------+-----------+-------------+-----------+------+---------------+-------------+----------------------+
|CUST000001 |42 |Male  |Pune   |Tier1    |salaried_senior|370941.36     |2021-10-17       |441             |1           |1         |1             |673.0      |6.5          |1          |1     |Salaried Senior|1043321819   |bbalay@ex

In [8]:
# Register all DataFrames as temporary SQL views
# This allows us to query them using SQL syntax
customers.createOrReplaceTempView("customers")
products.createOrReplaceTempView("products")
txn_history.createOrReplaceTempView("transactions_history")
txn_label.createOrReplaceTempView("transactions_label")
clv_labels.createOrReplaceTempView("clv_labels")

print("All tables registered as SQL views ✅")
print("You can now query them using spark.sql()\n")

# First SparkSQL query — basic sanity check
result = spark.sql("""
    SELECT
        city_tier,
        COUNT(*)                            AS total_customers,
        ROUND(AVG(monthly_income), 2)       AS avg_monthly_income,
        ROUND(AVG(cibil_score), 1)          AS avg_cibil_score,
        SUM(CASE WHEN kyc_complete = 1
                 THEN 1 ELSE 0 END)        AS kyc_complete_count
    FROM customers
    GROUP BY city_tier
    ORDER BY avg_monthly_income DESC
""")

print("Customer distribution by city tier:")
result.show(truncate=False)

# Second query — CLV distribution by bucket
clv_dist = spark.sql("""
    SELECT
        clv_bucket,
        COUNT(*)                              AS customers,
        ROUND(AVG(clv_next_12months), 2)      AS avg_clv,
        ROUND(MIN(clv_next_12months), 2)      AS min_clv,
        ROUND(MAX(clv_next_12months), 2)      AS max_clv
    FROM clv_labels
    GROUP BY clv_bucket
    ORDER BY avg_clv DESC
""")

print("CLV distribution by bucket:")
clv_dist.show(truncate=False)

All tables registered as SQL views ✅
You can now query them using spark.sql()

Customer distribution by city tier:
+---------+---------------+------------------+---------------+------------------+
|city_tier|total_customers|avg_monthly_income|avg_cibil_score|kyc_complete_count|
+---------+---------------+------------------+---------------+------------------+
|Metro    |17600          |311998.36         |668.3          |15654             |
|Tier1    |15012          |182674.21         |664.1          |13385             |
|Tier2    |12648          |123902.56         |665.8          |11296             |
|Tier3    |4990           |86084.85          |667.8          |4438              |
+---------+---------------+------------------+---------------+------------------+

CLV distribution by bucket:
+----------+---------+---------+---------+--------+
|clv_bucket|customers|avg_clv  |min_clv  |max_clv |
+----------+---------+---------+---------+--------+
|very_high |4260     |334160.13|100004.07|90

In [9]:
from pyspark.sql import functions as F

print("=" * 60)
print(" NULL ANALYSIS — All Tables")
print(" Using PySpark functions module (F)")
print("=" * 60)

def null_analysis(df, table_name):
    """
    Compute null count and null percentage for every column
    in a Spark DataFrame.

    In pandas we used df.isnull().sum().
    In PySpark we must compute this differently because
    Spark is distributed — we cannot iterate row by row.
    Instead we use F.sum(F.when(...).otherwise(...))
    which computes everything in one distributed pass.
    """
    total_rows = df.count()

    null_exprs = [
        F.sum(
            F.when(F.col(c).isNull(), 1).otherwise(0)
        ).alias(c)
        for c in df.columns
    ]

    null_counts = df.select(null_exprs).collect()[0]

    print(f"\n── {table_name.upper()} ({total_rows:,} rows) ──")
    print(f"  {'Column':<25} {'Nulls':>8} {'Null %':>8}")
    print(f"  {'-'*45}")
    for col_name in df.columns:
        n   = null_counts[col_name]
        pct = round(n / total_rows * 100, 2)
        flag = " ⚠️" if pct > 5 else ""
        print(f"  {col_name:<25} {n:>8,} {pct:>7.2f}%{flag}")

null_analysis(customers,   "customers")
null_analysis(products,    "products")
null_analysis(txn_history, "transactions_history (sample)")
null_analysis(clv_labels,  "clv_labels")

 NULL ANALYSIS — All Tables
 Using PySpark functions module (F)

── CUSTOMERS (50,250 rows) ──
  Column                       Nulls   Null %
  ---------------------------------------------
  customer_id                      0    0.00%
  age                              0    0.00%
  gender                       1,034    2.06%
  city                             0    0.00%
  city_tier                        0    0.00%
  segment                          0    0.00%
  monthly_income                   0    0.00%
  account_open_date                0    0.00%
  account_age_days                 0    0.00%
  kyc_complete                 1,522    3.03%
  pan_linked                       0    0.00%
  aadhaar_linked                   0    0.00%
  cibil_score                  4,008    7.98% ⚠️
  digital_score                    0    0.00%
  rm_assigned                      0    0.00%
  is_nri                           0    0.00%
  occupation                       0    0.00%
  phone                   

In [ ]:
from pyspark.sql.types import DateType, IntegerType, BooleanType

print("FIXING DATA TYPES\n")

# ── Fix customers table ───────────────────────────────────────
customers = customers.withColumn(
    "account_open_date",
    F.to_date(F.col("account_open_date"), "yyyy-MM-dd")
).withColumn(
    "kyc_complete",
    F.col("kyc_complete").cast(BooleanType())
).withColumn(
    "pan_linked",
    F.col("pan_linked").cast(BooleanType())
).withColumn(
    "aadhaar_linked",
    F.col("aadhaar_linked").cast(BooleanType())
).withColumn(
    "rm_assigned",
    F.col("rm_assigned").cast(BooleanType())
).withColumn(
    "is_nri",
    F.col("is_nri").cast(BooleanType())
)
customers.createOrReplaceTempView("customers")

# ── Fix transactions table ────────────────────────────────────
txn_history = txn_history.withColumn(
    "txn_date",
    F.to_date(F.col("txn_date"), "yyyy-MM-dd")
).withColumn(
    "is_international",
    F.col("is_international").cast(BooleanType())
)
txn_history.createOrReplaceTempView("transactions_history")

# ── Fix products table ────────────────────────────────────────
products = products.withColumn(
    "open_date",
    F.to_date(F.col("open_date"), "yyyy-MM-dd")
).withColumn(
    "is_active",
    F.col("is_active").cast(BooleanType())
)
products.createOrReplaceTempView("products")

print("Verifying date columns after fix:")
customers.select(
    "customer_id",
    "account_open_date"
).show(5)

txn_history.select(
    "txn_id",
    "txn_date",
    "amount"
).show(5)

customers.printSchema()
print("Schema after fixing customers:")

FIXING DATA TYPES

Verifying date columns after fix:
+-----------+-----------------+
|customer_id|account_open_date|
+-----------+-----------------+
| CUST000001|       2021-10-17|
| CUST000002|       2018-11-15|
| CUST000003|       2022-05-03|
| CUST000004|       2019-07-07|
| CUST000005|       2020-07-28|
+-----------+-----------------+
only showing top 5 rows
+-------------+----------+---------+
|       txn_id|  txn_date|   amount|
+-------------+----------+---------+
|TXN0000000001|2023-01-02|370941.36|
|TXN0000000002|2023-01-16| 22318.62|
|TXN0000000003|2023-01-23|   716.23|
|TXN0000000004|2023-01-19|  1343.14|
|TXN0000000005|2023-01-20|  2705.48|
+-------------+----------+---------+
only showing top 5 rows
Schema after fixing customers:
root
 |-- customer_id: string (nullable = true)
 |-- age: integer (nullable = true)
 |-- gender: string (nullable = true)
 |-- city: string (nullable = true)
 |-- city_tier: string (nullable = true)
 |-- segment: string (nullable = true)
 |-- mont

In [11]:
from pyspark.sql.window import Window

print("WINDOW FUNCTION — Monthly Spend Per Customer")
print("This is the most important PySpark concept for finance\n")

# Extract year and month from transaction date
txn_history = txn_history.withColumn(
    "year",  F.year(F.col("txn_date"))
).withColumn(
    "month", F.month(F.col("txn_date"))
)

# Define the window: partition by customer and month
customer_month_window = Window.partitionBy(
    "customer_id", "year", "month"
)

# Compute running balance within each customer-month
txn_history = txn_history.withColumn(
    "monthly_txn_count",
    F.count("txn_id").over(customer_month_window)
).withColumn(
    "monthly_debit_sum",
    F.sum(
        F.when(F.col("credit_debit") == "DR", F.col("amount"))
        .otherwise(0)
    ).over(customer_month_window)
)

print("Sample — customer monthly transaction summary:")
txn_history.select(
    "customer_id", "year", "month",
    "txn_id", "amount", "credit_debit",
    "monthly_txn_count", "monthly_debit_sum"
).filter(
    F.col("customer_id") == "CUST000001"
).orderBy("year", "month", "txn_date").show(20)

WINDOW FUNCTION — Monthly Spend Per Customer
This is the most important PySpark concept for finance

Sample — customer monthly transaction summary:
+-----------+----+-----+-------------+---------+------------+-----------------+------------------+
|customer_id|year|month|       txn_id|   amount|credit_debit|monthly_txn_count| monthly_debit_sum|
+-----------+----+-----+-------------+---------+------------+-----------------+------------------+
| CUST000001|2023|    1|TXN0000000020|   725.46|          DR|               29|124989.81000000003|
| CUST000001|2023|    1|TXN0000000001|370941.36|          CR|               29|124989.81000000003|
| CUST000001|2023|    1|TXN0000000013|   551.16|          DR|               29|124989.81000000003|
| CUST000001|2023|    1|TXN0000000028|  1194.61|          DR|               29|124989.81000000003|
| CUST000001|2023|    1|TXN0000000014| 14562.89|          DR|               29|124989.81000000003|
| CUST000001|2023|    1|TXN0000000021|   484.63|          DR

In [12]:
print("SPARKSQL WINDOW FUNCTIONS")
print("Same logic as Cell 10 but written in SQL\n")

# Re-register txn_history with new columns
txn_history.createOrReplaceTempView("transactions_history")

monthly_summary = spark.sql("""
    SELECT
        customer_id,
        year,
        month,
        COUNT(txn_id)                        AS txn_count,
        ROUND(SUM(CASE WHEN credit_debit = 'DR'
                       THEN amount ELSE 0 END), 2) AS total_debit,
        ROUND(SUM(CASE WHEN credit_debit = 'CR'
                       THEN amount ELSE 0 END), 2) AS total_credit,
        ROUND(AVG(amount), 2)                AS avg_txn_amount,
        COUNT(DISTINCT txn_type)             AS unique_txn_types,
        COUNT(DISTINCT merchant_category)    AS unique_categories
    FROM transactions_history
    GROUP BY customer_id, year, month
    ORDER BY customer_id, year, month
""")

monthly_summary.createOrReplaceTempView("monthly_summary")

print(f"Monthly summary rows: {monthly_summary.count():,}")
print(f"(50,000 customers × ~12 months = ~600,000 rows)\n")

print("Sample for one customer:")
spark.sql("""
    SELECT * FROM monthly_summary
    WHERE customer_id = 'CUST000001'
    ORDER BY year, month
""").show(truncate=False)

print("\nTop 5 customers by total annual debit spend:")
spark.sql("""
    SELECT
        customer_id,
        ROUND(SUM(total_debit), 2)   AS annual_spend,
        SUM(txn_count)               AS annual_txn_count
    FROM monthly_summary
    WHERE year = 2023
    GROUP BY customer_id
    ORDER BY annual_spend DESC
    LIMIT 5
""").show(truncate=False)

SPARKSQL WINDOW FUNCTIONS
Same logic as Cell 10 but written in SQL

Monthly summary rows: 581,938
(50,000 customers × ~12 months = ~600,000 rows)

Sample for one customer:
+-----------+----+-----+---------+-----------+------------+--------------+----------------+-----------------+
|customer_id|year|month|txn_count|total_debit|total_credit|avg_txn_amount|unique_txn_types|unique_categories|
+-----------+----+-----+---------+-----------+------------+--------------+----------------+-----------------+
|CUST000001 |2023|1    |29       |124989.81  |370941.36   |17101.07      |6               |11               |
|CUST000001 |2023|2    |36       |155518.11  |395707.38   |15311.82      |7               |12               |
|CUST000001 |2023|3    |29       |411103.31  |370941.36   |26967.06      |7               |12               |
|CUST000001 |2023|4    |39       |287901.79  |377022.97   |17049.35      |7               |12               |
|CUST000001 |2023|5    |24       |108806.99  |370941.36   

In [13]:
print("CACHING — How to Avoid Recomputing Expensive Results\n")

# Cache the monthly summary — we will use it many times
# in Feature Engineering
monthly_summary.cache()

# Force the cache to materialize by running an action
_ = monthly_summary.count()

print("monthly_summary is now cached in memory ✅")
print("Future queries on this DataFrame will NOT re-read")
print("from SQLite or recompute the aggregation.\n")

# Show what is cached
print("Spark Cache Status:")
print(f"  Is cached: {monthly_summary.is_cached}")

# Demonstrate the speed difference
import time

print("\nFirst query (reads from cache):")
t1 = time.time()
monthly_summary.filter(F.col("year") == 2023).count()
t2 = time.time()
print(f"  Time: {t2-t1:.3f} seconds")

print("\nUncache and re-query (reads from SQLite + recomputes):")
monthly_summary.unpersist()
t3 = time.time()
spark.sql("""
    SELECT COUNT(*) FROM (
        SELECT customer_id, year, month,
               COUNT(*) as n
        FROM transactions_history
        GROUP BY customer_id, year, month
    )
""").collect()
t4 = time.time()
print(f"  Time: {t4-t3:.3f} seconds")

print(f"\nSpeedup from caching: {(t4-t3)/(t2-t1):.1f}x")

# Re-cache for future use
monthly_summary.cache()
_ = monthly_summary.count()
print("\nRe-cached for future notebooks ✅")

CACHING — How to Avoid Recomputing Expensive Results

monthly_summary is now cached in memory ✅
Future queries on this DataFrame will NOT re-read
from SQLite or recompute the aggregation.

Spark Cache Status:
  Is cached: True

First query (reads from cache):
  Time: 0.499 seconds

Uncache and re-query (reads from SQLite + recomputes):
  Time: 62.110 seconds

Speedup from caching: 124.4x

Re-cached for future notebooks ✅


In [14]:
print("=" * 60)
print(" DUPLICATE DETECTION — PySpark Style")
print("=" * 60)

def check_duplicates(df, table_name, key_cols):
    """
    Detect duplicates in a Spark DataFrame.
    
    In pandas: df.duplicated().sum()
    In PySpark: groupBy all columns → count > 1
    
    Two types of duplicates:
    1. Exact row duplicates (every column identical)
    2. Primary key duplicates (key columns repeated,
       other columns may differ)
    """
    total = df.count()
    
    # Type 1 — exact row duplicates
    distinct = df.distinct().count()
    exact_dupes = total - distinct
    
    # Type 2 — primary key duplicates
    key_dupes = (
        df.groupBy(key_cols)
        .count()
        .filter(F.col("count") > 1)
        .count()
    )
    
    print(f"\n── {table_name.upper()} ──────────────────────")
    print(f"  Total rows          : {total:>10,}")
    print(f"  Distinct rows       : {distinct:>10,}")
    print(f"  Exact duplicates    : {exact_dupes:>10,}")
    print(f"  PK duplicates       : {key_dupes:>10,} "
          f"({', '.join(key_cols)})")

check_duplicates(customers,  "customers",  ["customer_id"])
check_duplicates(products,   "products",   ["product_id"])
check_duplicates(txn_history,"transactions_history", ["txn_id"])
check_duplicates(clv_labels, "clv_labels", ["customer_id"])

print("\nRemoving exact duplicates...")
customers   = customers.distinct()
products    = products.distinct()
txn_history = txn_history.distinct()
txn_label   = txn_label.distinct()
clv_labels  = clv_labels.distinct()

print("After deduplication:")
for name, df in [("customers", customers),
                 ("products",  products),
                 ("txn_history", txn_history),
                 ("clv_labels",  clv_labels)]:
    print(f"  {name:<20}: {df.count():>10,} rows")

 DUPLICATE DETECTION — PySpark Style

── CUSTOMERS ──────────────────────
  Total rows          :     50,250
  Distinct rows       :     50,000
  Exact duplicates    :        250
  PK duplicates       :        250 (customer_id)

── PRODUCTS ──────────────────────
  Total rows          :    124,865
  Distinct rows       :    124,865
  Exact duplicates    :          0
  PK duplicates       :          0 (product_id)

── TRANSACTIONS_HISTORY ──────────────────────
  Total rows          : 16,608,362
  Distinct rows       : 16,608,362
  Exact duplicates    :          0
  PK duplicates       :          0 (txn_id)

── CLV_LABELS ──────────────────────
  Total rows          :     50,000
  Distinct rows       :     50,000
  Exact duplicates    :          0
  PK duplicates       :          0 (customer_id)

Removing exact duplicates...
After deduplication:
  customers           :     50,000 rows
  products            :    124,865 rows
  txn_history         : 16,608,362 rows
  clv_labels          :

In [15]:
print("=" * 60)
print(" OUTLIER DETECTION — Percentile Analysis")
print("=" * 60)

# approxQuantile is the PySpark way to compute percentiles
# on large distributed datasets efficiently

print("\n── CUSTOMERS — Income and Age Outliers ──")
income_quantiles = customers.approxQuantile(
    "monthly_income",
    [0.01, 0.05, 0.25, 0.50, 0.75, 0.95, 0.99],
    relativeError=0.01
)
age_quantiles = customers.approxQuantile(
    "age",
    [0.01, 0.05, 0.25, 0.50, 0.75, 0.95, 0.99],
    relativeError=0.01
)

labels = ["P1","P5","P25","P50","P75","P95","P99"]
print(f"\n  {'Percentile':<8} {'Monthly Income':>16} {'Age':>8}")
print(f"  {'-'*36}")
for label, inc, age in zip(labels, income_quantiles, age_quantiles):
    print(f"  {label:<8} ₹{inc:>14,.2f} {age:>8.0f}")

# Flag impossible values
print("\n── IMPOSSIBLE VALUE CHECK ──")
impossible_age = customers.filter(
    (F.col("age") < 18) | (F.col("age") > 100)
).count()

impossible_income = customers.filter(
    F.col("monthly_income") <= 0
).count()

negative_amount = txn_history.filter(
    F.col("amount") < 0
).count()

print(f"  Age < 18 or > 100      : {impossible_age:>8,} rows")
print(f"  Monthly income <= 0    : {impossible_income:>8,} rows")
print(f"  Negative txn amounts   : {negative_amount:>8,} rows")

print("\n── TRANSACTIONS — Amount Distribution ──")
amount_quantiles = txn_history.approxQuantile(
    "amount",
    [0.25, 0.50, 0.75, 0.90, 0.95, 0.99, 0.999],
    relativeError=0.01
)
amt_labels = ["P25","P50","P75","P90","P95","P99","P99.9"]
for label, amt in zip(amt_labels, amount_quantiles):
    print(f"  {label:<8}: ₹{amt:>12,.2f}")

 OUTLIER DETECTION — Percentile Analysis

── CUSTOMERS — Income and Age Outliers ──

  Percentile   Monthly Income      Age
  ------------------------------------
  P1       ₹          3.27       -1
  P5       ₹      8,064.26       19
  P25      ₹     36,018.86       28
  P50      ₹     78,590.48       35
  P75      ₹    166,385.81       46
  P95      ₹    556,630.53       60
  P99      ₹  4,998,728.70      150

── IMPOSSIBLE VALUE CHECK ──
  Age < 18 or > 100      :    1,074 rows
  Monthly income <= 0    :        0 rows
  Negative txn amounts   :        0 rows

── TRANSACTIONS — Amount Distribution ──
  P25     : ₹      709.06
  P50     : ₹    1,580.45
  P75     : ₹    5,301.94
  P90     : ₹   19,648.66
  P95     : ₹   47,409.54
  P99     : ₹5,456,127.90
  P99.9   : ₹5,456,127.90


In [16]:
print("=" * 60)
print(" TABLE RELATIONSHIP VALIDATION")
print(" Verify referential integrity across tables")
print("=" * 60)

# Check 1 — every CLV label has a matching customer
clv_without_customer = clv_labels.join(
    customers.select("customer_id"),
    on="customer_id",
    how="left_anti"
).count()

print(f"\n CLV labels without matching customer : "
      f"{clv_without_customer:>8,}")
print(f"  (should be 0 — every label needs a customer)")

# Check 2 — every product belongs to a known customer
orphan_products = products.join(
    customers.select("customer_id"),
    on="customer_id",
    how="left_anti"
).count()

print(f"\n  Products without matching customer : "
      f"{orphan_products:>8,}")

# Check 3 — transaction coverage
# How many customers have at least 1 transaction?
customers_with_txns = txn_history.select(
    "customer_id"
).distinct().count()

print(f"\n  Customers with transactions   : "
      f"{customers_with_txns:>8,}")
print(f"  Total customers               : "
      f"{customers.count():>8,}")
print(f"  Coverage                      : "
      f"{customers_with_txns/customers.count()*100:>7.1f}%")

# Check 4 — product count per customer
print("\n  Product count distribution:")
spark.sql("""
    SELECT
        product_count,
        COUNT(*) AS customers
    FROM (
        SELECT customer_id, COUNT(*) AS product_count
        FROM products
        GROUP BY customer_id
    )
    GROUP BY product_count
    ORDER BY product_count
""").show(10)

 TABLE RELATIONSHIP VALIDATION
 Verify referential integrity across tables

 CLV labels without matching customer :        0
  (should be 0 — every label needs a customer)

  Products without matching customer :        0

  Customers with transactions   :   50,000
  Total customers               :   50,000
  Coverage                      :   100.0%

  Product count distribution:
+-------------+---------+
|product_count|customers|
+-------------+---------+
|            1|    11460|
|            2|    15121|
|            3|    13650|
|            4|     7146|
|            5|     2155|
|            6|      422|
|            7|       46|
+-------------+---------+



In [19]:
print("=" * 60)
print(" QUICK EDA SUMMARY — Key Business Metrics")
print("=" * 60)

# Metric 1 — Revenue potential by segment
print("\n1. CLV by Customer Segment:")
spark.sql("""
    SELECT
        c.segment,
        COUNT(*)                              AS customers,
        ROUND(AVG(l.clv_next_12months), 2)    AS avg_clv,
        ROUND(SUM(l.clv_next_12months), 2)    AS total_clv_portfolio,
        ROUND(MIN(l.clv_next_12months), 2)    AS min_clv,
        ROUND(MAX(l.clv_next_12months), 2)    AS max_clv
    FROM customers c
    JOIN clv_labels l ON c.customer_id = l.customer_id
    GROUP BY c.segment
    ORDER BY avg_clv DESC
""").show(truncate=False)

# Metric 2 — Transaction behavior by city tier
print("\n2. Transaction Behavior by City Tier:")
spark.sql("""
    SELECT
        c.city_tier,
        COUNT(DISTINCT t.customer_id)         AS active_customers,
        COUNT(t.txn_id)                       AS total_txns,
        ROUND(AVG(t.amount), 2)               AS avg_txn_amount,
        ROUND(SUM(CASE WHEN t.txn_type = 'UPI'
                       THEN 1 ELSE 0 END) * 100.0
              / COUNT(*), 1)                  AS upi_pct
    FROM transactions_history t
    JOIN customers c ON t.customer_id = c.customer_id
    GROUP BY c.city_tier
    ORDER BY avg_txn_amount DESC
""").show(truncate=False)

# Metric 3 — Product holding patterns
print("\n3. Active Product Holdings:")
spark.sql("""
    SELECT
        product_type,
        COUNT(*)                                            AS total_holdings,
        SUM(CASE WHEN CAST(is_active AS BOOLEAN) = TRUE
                 THEN 1 ELSE 0 END)                        AS active_count,
        ROUND(AVG(current_value), 2)                        AS avg_value,
        ROUND(SUM(current_value), 2)                        AS total_portfolio_value
    FROM products
    GROUP BY product_type
    ORDER BY total_portfolio_value DESC
""").show(truncate=False)

 QUICK EDA SUMMARY — Key Business Metrics

1. CLV by Customer Segment:
+---------------+---------+---------+-------------------+-------+---------+
|segment        |customers|avg_clv  |total_clv_portfolio|min_clv|max_clv  |
+---------------+---------+---------+-------------------+-------+---------+
|hni            |2465     |459867.86|1.13357427954E9    |0.03   |900000.0 |
|salaried_senior|4981     |76299.9  |3.8004978582E8     |0.65   |557354.78|
|self_employed  |7552     |35803.27 |2.703862951E8      |0.91   |454126.7 |
|salaried_mid   |14160    |26839.33 |3.8004490281E8     |0.0    |239245.15|
|retired        |2541     |7081.19  |1.799331236E7      |0.0    |80312.41 |
|salaried_entry |12515    |4991.85  |6.247304842E7      |0.0    |56491.72 |
|student        |6036     |181.54   |1095767.35         |0.0    |1673.84  |
+---------------+---------+---------+-------------------+-------+---------+


2. Transaction Behavior by City Tier:
+---------+----------------+----------+--------------